In [5]:
import numpy as np 
import pandas as pd 
from finalised_scripts import angle_between as angle
import moire_lattice_vector_finder as mlf 
import structure_writer as sw

In [6]:
file_1 = "mos2_mono_layer.vasp"
file_2 = "mos2_mono_layer.vasp"

In [7]:
lattice_vectors1, atom_type_arr1, dat1 = mlf.read_vasp(file_1)

lattice_vectors2, atom_type_arr2, dat2 = mlf.read_vasp(file_2)


In [8]:
candidate = pd.read_pickle('mos2_bilayer_candidates.pkl')

candidate.head()

,degree,n1,n2,n1p,n2p,vec_del,delta_angle,norm,A1,A2
0,21.8,1,3,3,2,0.001931,0.000007,14.508433,"[-1.58300104, 8.2255083654, 0.0]","[6.3319990486, 5.4836722436, 0.0]"


In [5]:
# pick you candidate
pick = candidate[candidate['degree']==21.8]

pick

,degree,n1,n2,n1p,n2p,vec_del,delta_angle,norm,A1,A2
0,21.8,1,3,3,2,0.001931,0.000007,14.508433,"[-1.58300104, 8.2255083654, 0.0]","[6.3319990486, 5.4836722436, 0.0]"


In [6]:
# Replication time 

replicate = int((pick['norm']/np.linalg.norm(lattice_vectors1['a']+lattice_vectors1['b']))+10)
replicate

/tmp/ipykernel_784534/1485595657.py:3: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  replicate = int((pick['norm']/np.linalg.norm(lattice_vectors1['a']+lattice_vectors1['b']))+10)


14

In [7]:
new_lattice_vectors1, new_total_atoms1, replicated_atom1, replicated_type_arr1 = sw.replicate_atoms(a = lattice_vectors1['a'],
                                                                                             b = lattice_vectors1['b'],
                                                                                             c = lattice_vectors1['c'],
                                                                                             atom_data = dat1,
                                                                                             atom_type_arr = atom_type_arr1,
                                                                                             natoms = len(dat1),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)

In [8]:
new_lattice_vectors2, new_total_atoms2, replicated_atom2, replicated_type_arr2 = sw.replicate_atoms(a = lattice_vectors2['a'],
                                                                                             b = lattice_vectors2['b'],
                                                                                             c = lattice_vectors2['c'],
                                                                                             atom_data = dat2,
                                                                                             atom_type_arr = atom_type_arr2,
                                                                                             natoms = len(dat2),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)

In [9]:
# new_lattice_vectors3, new_total_atoms3, replicated_atom3, replicated_type_arr3 = sw.replicate_atoms(a = lattice_vectors3['a'],
#                                                                                              b = lattice_vectors3['b'],
#                                                                                              c = lattice_vectors3['c'],
#                                                                                              atom_data = dat3,
#                                                                                              atom_type_arr = atom_type_arr3,
#                                                                                              natoms = len(dat3),
#                                                                                              na = 90,
#                                                                                              nb = 90,
#                                                                                              nc = 1)

In [10]:
# Create a DataFrame for positions
positions1_df = pd.DataFrame(replicated_atom1, columns=['x', 'y', 'z'])

# Add the atom types to the DataFrame
positions1_df['type'] = replicated_type_arr1


In [11]:
# Create a DataFrame for positions
positions2_df = pd.DataFrame(replicated_atom2, columns=['x', 'y', 'z'])

# Add the atom types to the DataFrame
positions2_df['type'] = replicated_type_arr2 
positions2_df['z'] = positions2_df['z'] + 6.0


In [12]:
# # Create a DataFrame for positions
# positions3_df = pd.DataFrame(replicated_atom3, columns=['x', 'y', 'z'])

# # Add the atom types to the DataFrame
# positions3_df['type'] = replicated_type_arr3
# positions3_df['z'] = positions3_df['z'] + 4.0*2 


In [13]:
theta1 = pick['degree'].values[0]

print(f"Twist Angle: {theta1}")

theta_rad = np.deg2rad(theta1)

print(f"Twist Angle in Radians: {theta_rad}")

rotation_matrix = np.array([[np.cos(theta_rad), -np.sin(theta_rad), 0],
                            [np.sin(theta_rad), np.cos(theta_rad), 0],
                            [0, 0, 1]])



# Rotate the positions in the upper layer
middle_layer_positions = positions2_df[['x', 'y', 'z']].values  # Extract x, y, z positions
rotated_positions = middle_layer_positions @ rotation_matrix.T  # Apply rotation

# Create a new DataFrame for the rotated positions
rotated_middle_layer = pd.DataFrame(rotated_positions, columns=['x', 'y', 'z'])

# Add the atom types back to the rotated DataFrame
rotated_middle_layer['type'] = positions2_df['type'].values

Twist Angle: 21.8
Twist Angle in Radians: 0.38048177693476387


In [14]:
# theta2 = 13.17+21.79

# print(f"Twist Angle: {theta2}")

# theta_rad = np.deg2rad(theta2)

# rotation_matrix = np.array([[np.cos(theta_rad), -np.sin(theta_rad), 0],
#                             [np.sin(theta_rad), np.cos(theta_rad), 0],
#                             [0, 0, 1]])



# # Rotate the positions in the upper layer
# upper_layer_positions = positions3_df[['x', 'y', 'z']].values  # Extract x, y, z positions
# rotated_positions = upper_layer_positions @ rotation_matrix.T  # Apply rotation

# # Create a new DataFrame for the rotated positions
# rotated_upper_layer = pd.DataFrame(rotated_positions, columns=['x', 'y', 'z'])

# # Add the atom types back to the rotated DataFrame
# rotated_upper_layer['type'] = positions3_df['type'].values

In [15]:
# Concatenate DataFrames vertically
concat_vertical = pd.concat([positions1_df, rotated_middle_layer], ignore_index=True)

In [16]:
# # Concatenate DataFrames vertically
# concat_vertical = pd.concat([positions1_df, rotated_middle_layer, rotated_upper_layer], ignore_index=True)

In [17]:
A1 = np.array(pick['A1'].values[0])
A2 = np.array(pick['A2'].values[0])
A3 = np.array([0, 0, 50.0])

print(f"A1:{A1}, A2:{A2}, A3:{A3}")
print(f"Len_A1: {np.linalg.norm(A1)}, Len_A2: {np.linalg.norm(A2)}")
print(f" Angle between A1 and A2: {angle.angle_between(A1,A2)}")

A1:[-1.58300104  8.22550837  0.        ], A2:[6.33199905 5.48367224 0.        ], A3:[ 0.  0. 50.]
Len_A1: 8.376447944200839, Len_A2: 8.376447530230244
 Angle between A1 and A2: 60.00000708460624


In [18]:
print("A1",A1)
print("A2",A2)
print("A1+A2",A1+A2)

A1 [-1.58300104  8.22550837  0.        ]
A2 [6.33199905 5.48367224 0.        ]
A1+A2 [ 4.74899801 13.70918061  0.        ]


In [19]:
# new moire vectors based on transformatin

V1 = np.linalg.norm(A1) * np.array([1,0,0])

A1_norm = A1 / np.linalg.norm(A1)

V2 = np.dot(A1_norm,A2)*np.array([1,0,0]) + (np.linalg.norm(np.cross(A1_norm,A2)))*np.array([0,1,0])


print(V1)
print(V2)
print(angle.angle_between(V1,V2))

[8.37644794 0.         0.        ]
[4.18822287 7.25421687 0.        ]
60.00000708460624


In [20]:
vertex = [ [0.0,0.0], [A1[0], A1[1]],  [(A1+A2)[0],(A1+A2)[1]], [A2[0],A2[1]]]
vertex 

[[0.0, 0.0],
 [-1.58300104, 8.2255083654],
 [4.7489980086, 13.709180609],
 [6.3319990486, 5.4836722436]]

In [21]:
from matplotlib.path import Path

# Define the polygon points
polygon_points = vertex

# Create a Path object for the polygon
polygon_path = Path(polygon_points)


# Function to select atoms within the polygon
def select_atoms_in_polygon_path(df, polygon_path):
    xy_points = df[['x','y']].values  # Extract x, y coordinates
    inside_mask = polygon_path.contains_points(xy_points)
    return df[inside_mask]

# Example usage:
selected_atoms_df = select_atoms_in_polygon_path(concat_vertical, polygon_path)

In [22]:
selected_atoms_df['type'].unique()

array(['Mo', 'S'], dtype=object)

In [23]:
unique_types = selected_atoms_df['type'].unique()

sort_order = {value: index for index, value in enumerate(unique_types)}

print(f"Sort Order of Elements: {sort_order}")

# Create a new column to map the sort order
selected_atoms_df['sort_key'] = selected_atoms_df['type'].map(sort_order)

# Sort the DataFrame by the new sort key and drop the sort_key column
sorted_df = selected_atoms_df.sort_values(by='sort_key').drop(columns='sort_key').copy()


Sort Order of Elements: {'Mo': 0, 'S': 1}


/tmp/ipykernel_784534/3018769804.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_atoms_df['sort_key'] = selected_atoms_df['type'].map(sort_order)


In [35]:
sorted_df.head()

,x,y,z,type
1218,-0.237450,1.416615,12.273334,Mo
3828,3.096157,8.670139,18.273334,Mo
3825,5.584180,6.712258,18.273334,Mo
3744,0.156571,7.494388,18.273334,Mo
3741,2.644594,5.536508,18.273334,Mo


In [36]:
for keys in sort_order:
    print(keys)
    print(len(sorted_df[sorted_df['type']==keys]))

Mo
14
S
28


In [40]:
element_count = " ".join(f"{len(sorted_df[sorted_df['type']==key])}" for key, _ in sort_order.items())
print(element_count)

14 28


In [41]:
def write_vasp_optimized(filename, A1, A2, A3, atom_data, sort_order):
    # Buffer for all the lines to write
    element_type = " ".join(f"{key}" for key, _ in sort_order.items())
    element_count = " ".join(f"{len(sorted_df[sorted_df['type']==key])}" for key, _ in sort_order.items())
    lines = [
        "System Generated by MLM \n",
        "1.0\n",
        f"{A1[0]} {A1[1]} {A1[2]}\n",
        f"{A2[0]} {A2[1]} {A2[2]}\n",
        f"{A3[0]} {A3[1]} {A3[2]}\n",
        f"{element_type}\n",
        f"{element_count} \n",
        "Cartesian\n"
    ]
    atomic_positions = atom_data[['x', 'y', 'z']]
    # Write the buffer to the file
    with open(filename, 'w') as file:
        file.writelines(lines)
        # Use to_csv for efficient DataFrame writing
        atomic_positions.to_csv(file, sep=' ', index=False, header=False, mode='a')

In [42]:
write_vasp_optimized("moire_mos2_21.8_rotated_test.vasp",A1,A2,A3, sorted_df, sort_order)

# LAMMPS Writer

In [59]:
selected_atoms_df.head()

,x,y,z,type,sort_key,numeric_id,mass
1218,-0.237450,1.416615,12.273334,Mo,0,2,95.940
1222,-0.237450,3.244506,10.696824,S,1,1,32.065
1223,-0.237450,3.244506,13.850458,S,1,3,32.065
1305,1.345549,4.158452,12.273334,Mo,0,2,95.940
1306,2.928549,3.244506,10.696824,S,1,1,32.065


In [57]:
len(selected_atoms_df)

42

## Set layer id as required by KC forcefield

In [58]:
unique_z = sorted(selected_atoms_df['z'].unique())

numeric_id_dict = {value: index+1 for index, value in enumerate(unique_z)}

print(f"Numeric Id: {numeric_id_dict}")

# Create a new column to map the sort order
selected_atoms_df['numeric_id'] = selected_atoms_df['z'].map(numeric_id_dict)




Numeric Id: {10.696823867: 1, 12.273333597: 2, 13.850457828: 3, 16.696823867: 4, 18.273333597: 5, 19.850457828: 6}


/tmp/ipykernel_784534/4183766918.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_atoms_df['numeric_id'] = selected_atoms_df['z'].map(numeric_id_dict)


## Set Masses Requires manual input

In [60]:
unique_element = selected_atoms_df['type'].unique()

print(f"Unique Elements: {unique_element}")



Unique Elements: ['Mo' 'S']


In [61]:
mass_type_dict = {'B': 10.811, 'C': 12.011, 'N': 14.007, 'O': 15.999, 
                  'S': 32.065, 'Mo': 95.94, 'W': 183.84, 'Se': 78.96, 
                  'Te': 127.6, 'Pb': 207.2, 'Bi': 208.980, 'Ti': 47.867}

In [62]:

selected_atoms_df['mass'] = selected_atoms_df['type'].map(mass_type_dict)

/tmp/ipykernel_784534/3153881743.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_atoms_df['mass'] = selected_atoms_df['type'].map(mass_type_dict)


In [63]:
selected_atoms_df.head()

,x,y,z,type,sort_key,numeric_id,mass
1218,-0.237450,1.416615,12.273334,Mo,0,2,95.940
1222,-0.237450,3.244506,10.696824,S,1,1,32.065
1223,-0.237450,3.244506,13.850458,S,1,3,32.065
1305,1.345549,4.158452,12.273334,Mo,0,2,95.940
1306,2.928549,3.244506,10.696824,S,1,1,32.065


In [64]:
sorted(selected_atoms_df['numeric_id'].unique())

[1, 2, 3, 4, 5, 6]

In [65]:
def write_lammps_data(filename, A1, A2, A3, atom_data, mass_type_dict):
    
    angle_A1 = angle.angle_between(A1,np.array([1,0,0]))
    angle_A2 = angle.angle_between(A2,np.array([1,0,0]))
    if angle_A1 < angle_A2:
        rot_rad = np.deg2rad(angle_A1)
    else:
        rot_rad = np.deg2rad(angle_A2)
    
    rotation_matrix = np.array([[np.cos(-rot_rad), -np.sin(-rot_rad), 0],
                            [np.sin(-rot_rad), np.cos(-rot_rad), 0],
                            [0, 0, 1]])

    A11 = np.round((rotation_matrix @ A1.T),4)
    A22 = np.round((rotation_matrix @ A2.T),4)
    
    type_counts = sorted(atom_data['numeric_id'].unique())
    lines = [
        "System Generated by MLM \n\n",
        f"{len(atom_data)} atoms\n",
        f"{int(atom_data['numeric_id'].nunique())} atom types\n\n",
        f"0.0 {max(A11[0],A22[0],A3[0])} xlo xhi\n"
        f"0.0 {max(A11[1],A22[1],A3[1])} ylo yhi\n",
        f"0.0 {max(A11[2],A22[2],A3[2])} zlo zhi\n",
        f"{sorted([A11[0],A22[0],A3[0]],reverse=True)[1]} 0.0 0.0 xy xz yz\n\n",
        "Masses\n\n",
    ]
    # Loop to print each element type and count 
    for key in type_counts:
        mass_value = atom_data.loc[atom_data['numeric_id'] == key, 'mass'].values[0] 
        lines.append(f"{key} {mass_value}\n") 
        
    lines.extend([ "\nAtoms #atomic \n\n" ])
    
    atomic_positions = atom_data[['x', 'y', 'z']].values
    rotated_atomic_pos = atomic_positions @ rotation_matrix.T
    rot_atomic_positions = pd.DataFrame(rotated_atomic_pos, columns=['x', 'y', 'z'])
    rot_atomic_positions.reset_index(inplace=True)
    rot_atomic_positions['id'] = rot_atomic_positions.index + 1
    rot_atomic_positions['numeric_id'] = atom_data['numeric_id'].values
    data = rot_atomic_positions[['id', 'numeric_id', 'x', 'y', 'z']]
    
    
    # Write the buffer to the file
    with open(filename, 'w') as file:
        file.writelines(lines)
        # Use to_csv for efficient DataFrame writing
        data.to_csv(file, sep=' ', index=False, header=False, mode='a')

In [66]:
write_lammps_data("lammps_test_mos2_21.8.data", A1, A2, A3, selected_atoms_df, mass_type_dict)

In [68]:
np.linalg.norm(A1)

8.376447944200839

In [69]:
np.linalg.norm(A2)

8.376447530230244